# GraphSAGE : Inductive Representation Learning on Large Graphs

In [ ]:
!pip install torch torch-geometric scikit-learn matplotlib -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 63.7 MB/s eta 0:00:00


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from sklearn.metrics import f1_score
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt
from collections import defaultdict
import random
from torch.utils.data import TensorDataset, DataLoader as TorchDataLoader
from torch_geometric.loader import DataLoader as GraphDataLoader
from sklearn.multioutput import MultiOutputClassifier

# Fixer les seeds pour reproductibilité
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

Device: cuda


## Chargement du dataset PPI


In [ ]:
from torch_geometric.datasets import PPI

# Dataset PPI
ppi_train_dataset = PPI(root='./data/PPI', split='train')
ppi_val_dataset = PPI(root='./data/PPI', split='val')
ppi_test_dataset = PPI(root='./data/PPI', split='test')

print("\nDAtaset PPI")
print(f"Train graphs: {len(ppi_train_dataset)}")
print(f"Val graphs: {len(ppi_val_dataset)}")
print(f"Test graphs: {len(ppi_test_dataset)}")
print(f"Features: {ppi_train_dataset[0].num_features}")
print(f"Labels (multi-label): {ppi_train_dataset[0].y.shape[1]}")

Extracting data/PPI/ppi.zip
Processing...



DAtaset PPI
Train graphs: 20
Val graphs: 2
Test graphs: 2
Features: 50
Labels (multi-label): 121


Done!


### Sampling du voisinage

In [ ]:
# Échantillonne uniformément un nombre fixe de voisins pour chaque nœud.

def sample_neighbors(edge_index, num_nodes, sample_size):

    # Construit une liste d'adjacence
    adj_list = defaultdict(list)
    for src, dst in edge_index.t().cpu().numpy():
        adj_list[src].append(dst)

    # Sampler pour chaque nœud
    sampled_neighbors = {}
    for node in range(num_nodes):
        neighbors = adj_list[node]
        if len(neighbors) >= sample_size:
            sampled = random.sample(neighbors, sample_size)
        else:
            # Si pas assez de voisins, on sample avec remplacement
            sampled = random.choices(neighbors, k=sample_size) if neighbors else [node] * sample_size
        sampled_neighbors[node] = sampled

    return sampled_neighbors

# Génère des random walks pour unsupervised loss.

def generate_random_walks(edge_index, num_nodes, walk_length=5, walks_per_node=3):

    adj_list = defaultdict(list)
    for src, dst in edge_index.t().cpu().numpy():
        adj_list[src].append(dst)

    pairs = []
    for node in range(num_nodes):
        for _ in range(walks_per_node):
            walk = [node]
            current = node
            for _ in range(walk_length - 1):
                neighbors = adj_list[current]
                if not neighbors:
                    break
                current = random.choice(neighbors)
                walk.append(current)

            # Crée des paires (node, context)
            for i, n in enumerate(walk):
                for j in range(max(0, i-2), min(len(walk), i+3)):
                    if i != j:
                        pairs.append((n, walk[j]))

    return pairs

## Aggregateurs

On implémente les aggregators suivants :
- Mean : Moyenne
- Pool : MLP + max-pooling
- GCN : Variante convolutional

In [ ]:
#Moyenne des embeddinds des voisins

class MeanAggregator(nn.Module):

    def forward(self, neighbor_features):
        # neighbor_features: [batch_size, num_neighbors, feature_dim]
        return torch.mean(neighbor_features, dim=1)

#MLP suivi de max pooling
class PoolAggregator(nn.Module):

    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(in_dim, out_dim),
            nn.ReLU()
        )

    def forward(self, neighbor_features):
        # Appliquer MLP sur chaque voisin
        batch_size, num_neighbors, feat_dim = neighbor_features.shape
        neighbor_features = neighbor_features.view(-1, feat_dim)
        transformed = self.mlp(neighbor_features)
        transformed = transformed.view(batch_size, num_neighbors, -1)
        # Max-pooling
        return torch.max(transformed, dim=1)[0]


class GCNAggregator(nn.Module):

    def forward(self, self_features, neighbor_features):
        # Concaténer self avec neighbors et faire moyenne
        combined = torch.cat([self_features.unsqueeze(1), neighbor_features], dim=1)
        return torch.mean(combined, dim=1)

## Implémentation de l'algorithme GraphSAGE

In [ ]:
#GraphSAGE avec K=2 layers
class GraphSAGE(nn.Module):

    def __init__(self, in_dim, hidden_dim, out_dim, aggregator_type='mean'):
        super().__init__()
        self.aggregator_type = aggregator_type

        # Couche 1
        if aggregator_type == 'mean':
            self.agg1 = MeanAggregator()
            self.fc1 = nn.Linear(in_dim * 2, hidden_dim)
        elif aggregator_type == 'pool':
            self.agg1 = PoolAggregator(in_dim, hidden_dim)
            self.fc1 = nn.Linear(in_dim + hidden_dim, hidden_dim)
        elif aggregator_type == 'gcn':
            self.agg1 = GCNAggregator()
            self.fc1 = nn.Linear(in_dim, hidden_dim)

        # Couche 2
        if aggregator_type == 'mean':
            self.agg2 = MeanAggregator()
            self.fc2 = nn.Linear(hidden_dim * 2, out_dim)
        elif aggregator_type == 'pool':
            self.agg2 = PoolAggregator(hidden_dim, out_dim)
            self.fc2 = nn.Linear(hidden_dim + out_dim, out_dim)
        elif aggregator_type == 'gcn':
            self.agg2 = GCNAggregator()
            self.fc2 = nn.Linear(hidden_dim, out_dim)

    def forward(self, x, neighbors_l1, neighbors_l2):

        batch_size = x.shape[0]

        # Couche 1
        # Récupére les features des voisins de niveau 2
        neighbor_feats_l2 = x[neighbors_l2]

        if self.aggregator_type == 'gcn':
            h1 = self.agg1(x, neighbor_feats_l2)
        else:
            agg_l2 = self.agg1(neighbor_feats_l2)
            h1 = torch.cat([x, agg_l2], dim=1)

        h1 = F.relu(self.fc1(h1))
        h1 = F.normalize(h1, p=2, dim=1)

        # Couche 2
        # Récupére embeddings des voisins de niveau 1
        neighbor_feats_l1 = h1[neighbors_l1]

        if self.aggregator_type == 'gcn':
            h2 = self.agg2(h1, neighbor_feats_l1)
        else:
            agg_l1 = self.agg2(neighbor_feats_l1)
            h2 = torch.cat([h1, agg_l1], dim=1)

        h2 = F.relu(self.fc2(h2))
        h2 = F.normalize(h2, p=2, dim=1)

        return h2

## Unsupervised Loss Function

Loss basée sur les random walks :
- Nœuds proches doivent avoir des embeddings similaires
- Nœuds éloignés doivent être distincts (negative sampling)

In [ ]:
class UnsupervisedLoss(nn.Module):

    def __init__(self, num_negative_samples=20):
        super().__init__()
        self.Q = num_negative_samples

    def forward(self, embeddings, u_indices, v_indices):

        # Récupére les embeddings pour le batch courant
        z_u = embeddings[u_indices]
        z_v = embeddings[v_indices]

        # Terme positif
        # On utilise le produit scalaire par paire
        pos_score = torch.sum(z_u * z_v, dim=1)
        pos_loss = -torch.log(torch.sigmoid(pos_score) + 1e-15).mean()

        # Terme negatif
        # Pour chaque u, on sample Q négatifs aléatoires
        num_nodes = embeddings.shape[0]
        neg_loss = 0

        for _ in range(self.Q):
            # Sampling aléatoire de nœuds négatifs
            neg_indices = torch.randint(0, num_nodes, size=u_indices.size(), device=u_indices.device)
            z_neg = embeddings[neg_indices]

            neg_score = torch.sum(z_u * z_neg, dim=1)
            neg_loss -= torch.log(torch.sigmoid(-neg_score) + 1e-15)

        # Moyenne sur le batch
        return torch.mean(pos_loss + neg_loss)

## Fonctions d'entrainements

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

def train_graphsage(model, data, pairs, num_epochs=20, lr=0.001, sample_sizes=[25, 10], batch_size=512):

    # Préparer le dataloader pour les paires
    u_list = [p[0] for p in pairs]
    v_list = [p[1] for p in pairs]

    dataset = TensorDataset(torch.tensor(u_list), torch.tensor(v_list))
    # On mélange les données à chaque epoch
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = UnsupervisedLoss(num_negative_samples=20)

    model.train()
    losses = []

    for epoch in range(num_epochs):
        # Pour simuler l'approche inductive, on re-échantillonne le voisinage une fois par époque (ou à chaque batch pour les puriste mais c'est lent)
        neighbors_l1 = sample_neighbors(data.edge_index, data.num_nodes, sample_sizes[0])
        neighbors_l2 = sample_neighbors(data.edge_index, data.num_nodes, sample_sizes[1])

        neighbors_l1_tensor = torch.tensor([neighbors_l1[i] for i in range(data.num_nodes)]).to(device)
        neighbors_l2_tensor = torch.tensor([neighbors_l2[i] for i in range(data.num_nodes)]).to(device)

        epoch_loss = 0

        for batch_u, batch_v in loader:
            batch_u = batch_u.to(device)
            batch_v = batch_v.to(device)

            optimizer.zero_grad()

            # Dans un cadre Big Data, on ne calculerait que les embeddings nécessaires
            embeddings = model(data.x, neighbors_l1_tensor, neighbors_l2_tensor)

            # Calcul de la loss sur le batch de paires uniquement
            loss = criterion(embeddings, batch_u, batch_v)

            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()

        avg_loss = epoch_loss / len(loader)
        losses.append(avg_loss)

        if (epoch + 1) % 5 == 0:
            print(f"Epoch {epoch+1}/{num_epochs}, Avg Loss: {avg_loss:.4f}")

    return losses

#Évalue les embeddings sur une tâche de classification.

def evaluate_node_classification(embeddings, labels, train_mask, test_mask):

    X_train = embeddings[train_mask].cpu().detach().numpy()
    y_train = labels[train_mask].cpu().numpy()
    X_test = embeddings[test_mask].cpu().detach().numpy()
    y_test = labels[test_mask].cpu().numpy()

    clf = LogisticRegression(max_iter=500, random_state=42)
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)

    micro_f1 = f1_score(y_test, y_pred, average='micro')
    macro_f1 = f1_score(y_test, y_pred, average='macro')

    return micro_f1, macro_f1


## PPI

PPI teste la capacité de généralisation et l'inductivité:  
entraîner sur certains graphes, tester sur des graphes totalement différents.

In [ ]:
class GraphSAGEClassifier(nn.Module):
    """Wrapper pour mode supervised avec classification head."""
    def __init__(self, graphsage, embed_dim, num_classes):
        super().__init__()
        self.graphsage = graphsage
        self.classifier = nn.Linear(embed_dim, num_classes)

    def forward(self, x, neighbors_l1, neighbors_l2):
        embeddings = self.graphsage(x, neighbors_l1, neighbors_l2)
        return self.classifier(embeddings)

## Fonction d'entrainement non supervisée

In [ ]:
def train_ppi_unsupervised(model, train_loader, num_epochs=30, lr=0.001):
    """
    Entraînement unsupervised avec random walks
    Recalcule les embeddings à chaque batch de paires pour éviter les conflits de gradient
    """
    from torch.utils.data import TensorDataset, DataLoader as PairLoader

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = UnsupervisedLoss(num_negative_samples=20)

    model.train()
    losses = []

    print(f"Entraînement unsupervised")

    for epoch in range(num_epochs):
        epoch_loss = 0
        num_batches = 0

        for batch_idx, batch in enumerate(train_loader):
            batch = batch.to(device)

            # Générer random walks pour ce graphe
            pairs = generate_random_walks(
                batch.edge_index,
                batch.num_nodes,
                walk_length=5,
                walks_per_node=3
            )

            if len(pairs) == 0:
                continue

            # Créer dataloader pour toutes les paires
            u_list = [p[0] for p in pairs]
            v_list = [p[1] for p in pairs]
            pair_dataset = TensorDataset(
                torch.tensor(u_list),
                torch.tensor(v_list)
            )
            pair_loader = PairLoader(pair_dataset, batch_size=512, shuffle=True)

            # Sample neighbors une fois pour ce graphe
            neighbors_l1 = sample_neighbors(batch.edge_index, batch.num_nodes, 25)
            neighbors_l2 = sample_neighbors(batch.edge_index, batch.num_nodes, 10)
            neighbors_l1_tensor = torch.tensor([neighbors_l1[i] for i in range(batch.num_nodes)]).to(device)
            neighbors_l2_tensor = torch.tensor([neighbors_l2[i] for i in range(batch.num_nodes)]).to(device)

            for pair_batch_u, pair_batch_v in pair_loader:
                pair_batch_u = pair_batch_u.to(device)
                pair_batch_v = pair_batch_v.to(device)

                optimizer.zero_grad()

                # Recalculer les embeddings à chaque fois
                embeddings = model(batch.x, neighbors_l1_tensor, neighbors_l2_tensor)

                # Loss sur ce batch de paires
                loss = criterion(embeddings, pair_batch_u, pair_batch_v)

                loss.backward()
                optimizer.step()

                epoch_loss += loss.item()
                num_batches += 1

        avg_loss = epoch_loss / max(num_batches, 1)
        losses.append(avg_loss)

        if (epoch + 1) % 5 == 0:
            print(f"  Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss:.4f}")

    return losses

## Fonction d'entrainement supervisée

In [ ]:
def train_ppi_supervised(model, train_loader, num_epochs=10, lr=0.005):

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCEWithLogitsLoss()

    model.train()
    losses = []

    print(f"Entraînement supervised")

    for epoch in range(num_epochs):
        epoch_loss = 0
        for batch in train_loader:
            batch = batch.to(device)

            neighbors_l1 = sample_neighbors(batch.edge_index, batch.num_nodes, 25)
            neighbors_l2 = sample_neighbors(batch.edge_index, batch.num_nodes, 10)
            neighbors_l1_tensor = torch.tensor([neighbors_l1[i] for i in range(batch.num_nodes)]).to(device)
            neighbors_l2_tensor = torch.tensor([neighbors_l2[i] for i in range(batch.num_nodes)]).to(device)

            logits = model(batch.x, neighbors_l1_tensor, neighbors_l2_tensor)
            loss = criterion(logits, batch.y.float())

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()

        avg_loss = epoch_loss / len(train_loader)
        losses.append(avg_loss)

        if (epoch + 1) % 5 == 0:
            print(f"  Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss:.4f}")

    return losses

## Fonctions d'entrainement

In [ ]:
def evaluate_ppi(model, test_loader, use_classifier=True):
    model.eval()
    all_preds = []
    all_labels = []
    all_embeddings = []

    with torch.no_grad():
        for batch in test_loader:
            batch = batch.to(device)

            neighbors_l1 = sample_neighbors(batch.edge_index, batch.num_nodes, 25)
            neighbors_l2 = sample_neighbors(batch.edge_index, batch.num_nodes, 10)
            neighbors_l1_tensor = torch.tensor([neighbors_l1[i] for i in range(batch.num_nodes)]).to(device)
            neighbors_l2_tensor = torch.tensor([neighbors_l2[i] for i in range(batch.num_nodes)]).to(device)

            if use_classifier:
                logits = model(batch.x, neighbors_l1_tensor, neighbors_l2_tensor)
                preds = (torch.sigmoid(logits) > 0.5).float()
            else:
                embeddings = model.graphsage(batch.x, neighbors_l1_tensor, neighbors_l2_tensor)
                all_embeddings.append(embeddings.cpu())

            all_labels.append(batch.y.cpu())
            if use_classifier:
                all_preds.append(preds.cpu())

    all_labels = torch.cat(all_labels, dim=0).numpy()

    if use_classifier:
        all_preds = torch.cat(all_preds, dim=0).numpy()
        micro_f1 = f1_score(all_labels, all_preds, average='micro')
    else:
        micro_f1 = 0.0

    return micro_f1


def evaluate_ppi_unsupervised(graphsage_model, train_loader, test_loader):

    graphsage_model.eval()
    train_embeddings = []
    train_labels = []

    with torch.no_grad():
        for batch in train_loader:
            batch = batch.to(device)
            neighbors_l1 = sample_neighbors(batch.edge_index, batch.num_nodes, 25)
            neighbors_l2 = sample_neighbors(batch.edge_index, batch.num_nodes, 10)
            n1 = torch.tensor([neighbors_l1[i] for i in range(batch.num_nodes)]).to(device)
            n2 = torch.tensor([neighbors_l2[i] for i in range(batch.num_nodes)]).to(device)

            embeddings = graphsage_model(batch.x, n1, n2)
            train_embeddings.append(embeddings.cpu())
            train_labels.append(batch.y.cpu())

    train_embeddings = torch.cat(train_embeddings, dim=0).numpy()
    train_labels = torch.cat(train_labels, dim=0).numpy()

    test_embeddings = []
    test_labels = []

    with torch.no_grad():
        for batch in test_loader:
            batch = batch.to(device)
            neighbors_l1 = sample_neighbors(batch.edge_index, batch.num_nodes, 25)
            neighbors_l2 = sample_neighbors(batch.edge_index, batch.num_nodes, 10)
            n1 = torch.tensor([neighbors_l1[i] for i in range(batch.num_nodes)]).to(device)
            n2 = torch.tensor([neighbors_l2[i] for i in range(batch.num_nodes)]).to(device)

            embeddings = graphsage_model(batch.x, n1, n2)
            test_embeddings.append(embeddings.cpu())
            test_labels.append(batch.y.cpu())

    test_embeddings = torch.cat(test_embeddings, dim=0).numpy()
    test_labels = torch.cat(test_labels, dim=0).numpy()

    clf = MultiOutputClassifier(LogisticRegression(max_iter=500, random_state=42))
    clf.fit(train_embeddings, train_labels)

    test_preds = clf.predict(test_embeddings)
    micro_f1 = f1_score(test_labels, test_preds, average='micro')

    return micro_f1

In [ ]:
train_loader = GraphDataLoader(ppi_train_dataset, batch_size=1, shuffle=True)
val_loader = GraphDataLoader(ppi_val_dataset, batch_size=1, shuffle=False)
test_loader = GraphDataLoader(ppi_test_dataset, batch_size=1, shuffle=False)


print("BASELINE : RAW FEATURES (sans GraphSAGE)")

train_features = []
train_labels = []
for batch in train_loader:
    train_features.append(batch.x.cpu())
    train_labels.append(batch.y.cpu())

train_features = torch.cat(train_features, dim=0).numpy()
train_labels = torch.cat(train_labels, dim=0).numpy()

test_features = []
test_labels = []
for batch in test_loader:
    test_features.append(batch.x.cpu())
    test_labels.append(batch.y.cpu())

test_features = torch.cat(test_features, dim=0).numpy()
test_labels = torch.cat(test_labels, dim=0).numpy()

baseline_clf = MultiOutputClassifier(LogisticRegression(max_iter=500, random_state=42))
baseline_clf.fit(train_features, train_labels)
baseline_preds = baseline_clf.predict(test_features)

baseline_micro_f1 = f1_score(test_labels, baseline_preds, average='micro')
print(f"Micro-F1: {baseline_micro_f1:.4f}")

results_ppi = {
    'baseline': {'unsup_f1': baseline_micro_f1, 'sup_f1': baseline_micro_f1}
}

BASELINE : RAW FEATURES (sans GraphSAGE)
Micro-F1: 0.4317


In [ ]:
hidden_dim = 256
unsupervised_epochs = 2
supervised_epochs = 10

for agg_type in ['mean', 'pool', 'gcn']:
    print(f"GraphSAGE-{agg_type.upper()}")

    # UNSUPERVISED
    print(f"-> UNSUPERVISED")

    graphsage_unsup = GraphSAGE(
        in_dim=ppi_train_dataset[0].num_features,
        hidden_dim=hidden_dim,
        out_dim=hidden_dim,
        aggregator_type=agg_type
    ).to(device)

    losses_unsup = train_ppi_unsupervised(
        graphsage_unsup,
        train_loader,
        num_epochs=unsupervised_epochs,
        lr=0.0001
    )

    unsup_f1 = evaluate_ppi_unsupervised(graphsage_unsup, train_loader, test_loader)
    print(f"   Unsupervised F1: {unsup_f1:.4f}")


    # SUPERVISED
    print(f"-> SUPERVISED")

    graphsage_sup = GraphSAGE(
        in_dim=ppi_train_dataset[0].num_features,
        hidden_dim=256,
        out_dim=256,
        aggregator_type=agg_type
    ).to(device)

    model_sup = GraphSAGEClassifier(
        graphsage_sup,
        embed_dim=hidden_dim,
        num_classes=ppi_train_dataset[0].y.shape[1]
    ).to(device)

    losses_sup = train_ppi_supervised(
        model_sup,
        train_loader,
        num_epochs=supervised_epochs,
        lr=0.005
    )

    sup_f1 = evaluate_ppi(model_sup, test_loader, use_classifier=True)
    print(f"   Supervised F1: {sup_f1:.4f}")

    results_ppi[agg_type] = {
        'unsup_f1': unsup_f1,
        'sup_f1': sup_f1,
        'losses_unsup': losses_unsup,
        'losses_sup': losses_sup
    }

GraphSAGE-MEAN
-> UNSUPERVISED
Entraînement unsupervised (2 epochs)...
   Unsupervised F1: 0.4179
-> SUPERVISED
Entraînement supervised (10 epochs)...
  Epoch 5/10, Loss: 0.5213
  Epoch 10/10, Loss: 0.4791
   Supervised F1: 0.5665
GraphSAGE-POOL
-> UNSUPERVISED
Entraînement unsupervised (2 epochs)...
   Unsupervised F1: 0.3983
-> SUPERVISED
Entraînement supervised (10 epochs)...
  Epoch 5/10, Loss: 0.5455
  Epoch 10/10, Loss: 0.5043
   Supervised F1: 0.5411
GraphSAGE-GCN
-> UNSUPERVISED
Entraînement unsupervised (2 epochs)...
   Unsupervised F1: 0.3938
-> SUPERVISED
Entraînement supervised (10 epochs)...
  Epoch 5/10, Loss: 0.5456
  Epoch 10/10, Loss: 0.5256
   Supervised F1: 0.4628


## Tableau comparatif

In [ ]:
print(f"\n{'Method':<25} {'Unsupervised F1':<15} {'Supervised F1':<15}")

print(f"{'Raw features':<25} {results_ppi['baseline']['unsup_f1']:<15.3f} {results_ppi['baseline']['sup_f1']:<15.3f}")

for agg in ['gcn', 'mean', 'pool']:
    paper_key_mapping = {
        'gcn': 'GraphSAGE-GCN',
        'mean': 'GraphSAGE-mean',
        'pool': 'GraphSAGE-pool'
    }
    paper_key = paper_key_mapping[agg]
    method_name = f"GraphSAGE-{agg.upper()}"

    print(f"{method_name:<25} "
          f"{results_ppi[agg]['unsup_f1']:<15.3f} "
          f"{results_ppi[agg]['sup_f1']:<15.3f}")


Method                    Unsupervised F1 Supervised F1  
Raw features              0.432           0.432          
GraphSAGE-GCN             0.394           0.463          
GraphSAGE-MEAN            0.418           0.567          
GraphSAGE-POOL            0.398           0.541          
